# Five animals with a CNN comparison

## 1. Objective
Compare the Logistic Regression classifiers with a lightweight CNN. The original small CNN and its results are retained. This comparison freezes a pretrained convolutional backbone and trains a small five-class neural head.

PyTorch computes gradients and trains the head; torchvision supplies the pretrained backbone and transforms. pandas preserves image IDs, and Matplotlib draws learning curves. This notebook is a comparison; binary Sigmoid and multinomial Softmax Logistic Regression remain the project's main focus.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd() if (Path.cwd() / 'common.py').exists() else Path.cwd().parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from common import LABELS, SEED, result_summary, prediction_examples, comparison_table
OUT = ROOT / 'outputs/improved'
plt.rcParams.update({'figure.dpi':110,'font.size':10})


## 2. Train the comparison model
The head has 576 inputs, 128 hidden units and five output scores. Training uses random resized crops and horizontal flips; validation and inference use the documented centre crop. Class weights come only from training counts. The frozen backbone stays in evaluation mode so its batch-normalization statistics do not learn from validation or test images.

Adam trains for at most eight epochs at learning rate 0.001. Validation loss selects the checkpoint, with early-stopping patience three. We keep the backbone frozen for this focused comparison. Detailed epoch records stay in `outputs/improved/transfer_history.csv`.

In [ ]:
from train_improved import train_transfer
RETRAIN = False
if RETRAIN or not (ROOT / 'models/transfer_cnn.pt').exists():
    train_transfer()
history = pd.read_csv(OUT / 'transfer_history.csv')
fig,axes = plt.subplots(1,2,figsize=(10,3.4))
for ax,metric in zip(axes,['loss','accuracy']):
    ax.plot(history.epoch,history[f'train_{metric}'],label='Training')
    ax.plot(history.epoch,history[f'validation_{metric}'],label='Validation')
    ax.set(xlabel='Epoch',ylabel=metric.capitalize()); ax.legend()
axes[1].yaxis.set_major_formatter(plt.matplotlib.ticker.PercentFormatter(1))
fig.tight_layout(); fig.savefig(OUT / 'transfer_learning_curves.png',dpi=160); plt.show()


## 3. Evaluate the selected checkpoint
The CNN and five-class Logistic Regression use the same held-out image IDs and labels. Accuracy, precision, recall and F1 are shown as percentages. Macro-F1 gives each class equal weight.

In [ ]:
a = pd.read_csv(OUT / 'softmax_features_predictions.csv')
b = pd.read_csv(OUT / 'transfer_cnn_predictions.csv')
assert a[['image_id','truth']].equals(b[['image_id','truth']])
result_summary('transfer_cnn', LABELS)


## 4. Compare the models
The two-class and five-class tasks have different test images and difficulty. Compare improvements within each task. Details such as timing and model sizes remain in supporting records.

In [ ]:
display(comparison_table())

## 5. Inspect example predictions
These are dataset-labelled examples, including errors. An unsupported animal will still receive one of the five labels.

In [ ]:
prediction_examples('transfer_cnn', LABELS)

## 6. Evaluation limits and a separate check
The original Animals-10 test split was inspected before this improvement study. Its scores are historical comparisons, not an untouched final assessment.

A separate licensed iNaturalist audit, selected and visually reviewed before inference, gave **53.49% accuracy** and **52.46% macro-F1** on 86 animal photographs. The full 100-observation audit and the photo eligibility decisions are also saved. No external result was used for tuning or sample replacement.

This small sample includes distant, dark and partly hidden animals. It shows a substantial domain gap. Contributor labels and dates, missing original source identities and incomplete duplicate detection limit any independence guarantee. These models are teaching classifiers; their historical scores do not establish reliable performance on arbitrary new photos.